# Notebook 03 — Yield Curve Preprocessing

Handles: Yield Curve Cleaning, Maturity Handling, Risk-Free Rate Creation, r

**Method:** Linear interpolation between adjacent standard maturities on the DTE axis.

**Input :** `data/raw/yield_curve/treasury_yield_curve.csv`  
**Output:** `data/processed/intermediate/yield_curve_cleaned.csv`

In [ ]:
import sys, os
sys.path.insert(0, os.path.join('..', 'src', 'preprocessing'))
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
print('Libraries loaded.')

## Step 1 — Run the Yield Curve Preprocessing Module

In [ ]:
import yield_curve_preprocessing

df_yield = yield_curve_preprocessing.clean_yield_curve(verbose=True)


## Step 2 — Inspect the Cleaned Yield Curve

In [ ]:
print('Shape:', df_yield.shape)
df_yield.head(5)


## Step 3 — Maturity Matching Method

**DOCUMENTED DECISION:**

1. The yield curve provides standard maturities: 1mo, 2mo, 3mo, 4mo, 6mo, 1yr, 2yr, 3yr, 5yr, 7yr, 10yr, 20yr, 30yr.
2. Each maturity label is converted to approximate calendar days (e.g. `1 mo` = 30 days, `1 yr` = 365 days).
3. For each option with a given `days_to_expiration`, `numpy.interp` performs linear interpolation between the two
   nearest maturity points.
4. Rates are in % in raw data — divided by 100 to get decimal form for Black-Scholes.
5. DTEs below the shortest available maturity use the shortest rate (floor clamping).
6. DTEs above the longest available maturity use the longest rate (ceiling clamping).

In [ ]:
# Show maturity-to-days mapping used
print('Maturity -> Days mapping:')
for k, v in yield_curve_preprocessing.MATURITY_DAYS.items():
    print(f'  {k:6s} = {v} days')


In [ ]:
# Demonstrate interpolation for the option date 2023-08-25 with various DTEs
date = pd.Timestamp('2023-08-25')
test_dtes = [5, 10, 15, 30, 60, 90, 120]
print(f'Interpolated r values for {date.date()}:')
for dte in test_dtes:
    r = yield_curve_preprocessing.interpolate_r(date, dte, df_yield)
    print(f'  DTE={dte:3d} days -> r = {r:.4f} ({r*100:.4f}%)')


## Step 4 — Yield Curve Visualisation (2023)

In [ ]:
df_2023 = df_yield[df_yield['date'].dt.year == 2023].copy()
mat_cols = [c for c in df_yield.columns if c != 'date']

# Plot the yield curve for the option date
row = df_yield[df_yield['date'] == pd.Timestamp('2023-08-25')]
if not row.empty:
    row = row.iloc[0]
    days_arr = [yield_curve_preprocessing.MATURITY_DAYS[c] for c in mat_cols if c in yield_curve_preprocessing.MATURITY_DAYS and not pd.isna(row[c])]
    rates_arr = [row[c]*100 for c in mat_cols if c in yield_curve_preprocessing.MATURITY_DAYS and not pd.isna(row[c])]
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(days_arr, rates_arr, 'o-', color='steelblue')
    ax.set_title('US Treasury Par Yield Curve — 2023-08-25')
    ax.set_xlabel('Days to Maturity')
    ax.set_ylabel('Yield (%)')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join('..', 'outputs', 'figures', '03_yield_curve.png'), dpi=100)
    plt.show()
    print('Plot saved.')


## Summary

- Yield curve raw dataset: 8,507 rows (1990-01-02 to 2023-12-29)
- r assigned via linear interpolation between adjacent standard maturity points
- Rates converted from % to decimal (divide by 100)
- Output saved to `data/processed/intermediate/yield_curve_cleaned.csv`